# PPIFlow VHH Design Pipeline

In [0]:
%pip install torch-scatter==2.1.2 
#%pip install nvidia-cublas-cu12==12.5.3.2 --index-url=https://pypi.org/simple cuequivariance-torch cuequivariance-ops-torch-cu12

In [0]:
%restart_python

In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
import sys
sys.path.append("/Workspace/Users/karthik.raj@bio-techne.com/ScoringPhysics")

## Step 1: Generate VHH Backbone via PPIFlow

In [0]:
#@title 1. PPIFlow backbone generation

#@markdown --- **Configuration Parameters** ---

#@markdown **Antigen PDB Path:** Path to the antigen PDB file in Google Drive. This file contains the protein the nanobody will bind to.
antigen_pdb = "../targets/Renamed_Adaptyv_RBX1.pdb" #@param {type:"string"}
#@markdown **Framework PDB Path:** Path to the framework PDB file in Google Drive. This file provides the structural scaffold for the nanobody.
framework_pdb = "../framework/8coh_nanobody_framework.pdb" #@param {type:"string"}
#@markdown **Antigen Chain ID:** The specific chain identifier within the antigen PDB file that the nanobody targets.
antigen_chain = "B" #@param {type:"string"}
#@markdown **Heavy Chain ID:** The chain identifier for the heavy chain of the nanobody within the Framework PDB.
heavy_chain = "A" #@param {type:"string"}
#@markdown **Specified Hotspots:** A comma-separated list of residues on the antigen that are crucial for binding. These are usually in the format 'ChainIDResidueNumber'.
specified_hotspots = "B33,B34,B35,B36,B37,B38,B39,B43,B44,B45,B46,B47,B48,B49,B50,B86,B87,B88,B89,B90,B91,B92" #@param {type:"string"}
#@markdown **CDR Lengths:** Specifies the desired lengths for the Complementarity Determining Regions (CDRs). Format: 'CDRName,MinLength-MaxLength'.
cdr_length = "CDRH1,8-8,CDRH2,8-8,CDRH3,9-21" #@param {type:"string"}
#@markdown **Samples Per Target:** The number of nanobody backbone samples to generate for each target.
samples_per_target = 7  #@param {type:"integer"}
#@markdown **Config Path:** Path to the YAML configuration file for the PPIFlow model.
config_path = "../configs/inference_nanobody.yaml" #@param {type:"string"}
#@markdown **Model Weights Path:** Path to the pre-trained model weights (.ckpt file) for the nanobody generation model.
model_weights = "/Volumes/sandbox/model_weights/protein_hunter/nanobody.ckpt" #@param {type:"string"}
#@markdown **Output Name Prefix:** A prefix to use for the names of the generated output files.
PREFIX = "RBX1"
framework_id = (framework_pdb.split('/')[-1]).split('_')[0]
name = f"Mar25_FMWK{framework_id}_{PREFIX}_{samples_per_target}_bbones" #@param {type:"string"}
#@markdown **Output Directory:** Directory where the generated PDB files will be saved.
output_dir =f"/tmp/{name}" #@param {type:"string"}


# Run via subprocess
import subprocess
subprocess.run(["python", "../sample_antibody_nanobody.py", 
                "--antigen_pdb", antigen_pdb, 
                "--framework_pdb", framework_pdb, 
                "--antigen_chain", antigen_chain, 
                "--heavy_chain", heavy_chain, 
                "--specified_hotspots", specified_hotspots, 
                "--cdr_length", cdr_length, 
                "--samples_per_target", str(samples_per_target), 
                "--config", config_path, 
                "--model_weights", model_weights, 
                "--output_dir", output_dir, 
                "--name", name])


In [0]:
import shutil
# 1. Define paths
job_name = output_dir.split("/")[-1]
volume_save_path = f'/Volumes/sandbox/denovotrial/ppiflow/{job_name}'

# 2. Use shutil.copytree instead of dbutils
# dirs_exist_ok=True allows it to overwrite/merge if the folder already exists
shutil.copytree(output_dir, volume_save_path, dirs_exist_ok=True)

In [0]:
#@title Display backbone
import py3Dmol

# 1. 读取 PDB 文件
pdb_path = f"{volume_save_path}/{name}_0.pdb" #@param {type:"string"}

with open(pdb_path, 'r') as f:
    pdb_content = f.read()

# 2. 创建查看器
# 设置背景为白色以增加对比度，方便观察 B-factor 颜色
view = py3Dmol.view(width=800, height=600)
view.addModel(pdb_content, 'pdb')

# 3. 为 A 链和 C 链设置基于 B-factor 的着色样式
# 'prop': 'b' 指代 B-factor
# 'gradient': 'roygb' (红-橙-黄-绿-蓝) 是经典的冷热图映射
# 通常高 B-factor（不稳定/柔性）显示为红色，低 B-factor（稳定）显示为蓝色
style_config = {
    'cartoon': {
        'colorscheme': {
            'prop': 'b',
            'gradient': 'roygb',
            'min': 0,    # 可选：手动设置 B-factor 范围以统一色阶
            'max': 5   # 可选：根据实际数据调整
        }
    }
}

# 分别应用到两条链
view.setStyle({'chain': 'A'}, style_config)
view.setStyle({'chain': 'B'}, style_config)

# 4. 辅助视觉效果
view.setBackgroundColor('white') # 设置白色背景
view.zoomTo()                    # 自动缩放以适应模型

# 5. 显示
view.show()

In [0]:
import py2Dmol
from StrucTools import *

def visualize_structure(structure_path):
    """
    Visualize a protein structure using py2Dmol.
    """
    viewer = py2Dmol.view()
    viewer.add_pdb(structure_path)
    viewer.show()
visualize_structure(f"{volume_save_path}/{name}_0.pdb")

### Filter for Designs with CDR Interface Ratio >= 0.6

In [0]:
import pandas as pd
import os, shutil
import numpy as np
df = pd.read_csv(f"{volume_save_path}/sample_metrics.csv")
df_temp = df[['pdb_path', 'sample', 'interface_residues', 'cdr_interface_residues', 'cdr_interface_ratio']]
df_temp['num_interface_residues'] = df_temp['interface_residues'].apply(lambda x: len(x.split(',')))
df_temp['num_cdr_interface_residues'] = df_temp['cdr_interface_residues'].apply(lambda x: len(x.split(',')))
df_temp['calc_cdr_interface_ratio'] = np.round(df_temp['num_cdr_interface_residues']/df_temp['num_interface_residues'], decimals= 3)
df_temp

In [0]:
import os
filtered_dir = f"{volume_save_path}/filtered_links"
os.makedirs(filtered_dir, exist_ok=True)

df_filtered = df[df['cdr_interface_ratio'] >= 0.6]
# Save Files to the filtered directory
for i, row in df_filtered.iterrows():
    pdb_path = f"{volume_save_path}/{row['sample']}"
    pdb_filtered_path = f"{filtered_dir}/{row['sample']}"
    shutil.copy(pdb_path, pdb_filtered_path)

## Step 2: Generate Seqs that will fold into VHH Backbones that pass CDR Interface Ratio > 0.6

In [0]:
#@title 2. MPNN Squence Design
import subprocess

abmpnn_checkpoint ="/Volumes/sandbox/model_weights/protein_hunter"
model_name = "abmpnn"
mpnn_dir =f"{volume_save_path}/nanobody_mpnn"
position_fixed =f"{filtered_dir}/fixed_positions.csv"
chains_to_design ="A"
num_seq_per_target=8
sampling_temp = 0.5
seed = 37
fixed_positions_input_folder_path = filtered_dir
folder_with_pdbs = filtered_dir

# Run setup script
subprocess.run(['python', '../demo_scripts/make_fix_csv.py', fixed_positions_input_folder_path])

# Run actual protein_mpnn_run script
subprocess.run(["python", "../ProteinMPNN/protein_mpnn_run.py", 
                "--path_to_model_weights", abmpnn_checkpoint, 
                "--model_name", model_name, 
                "--folder_with_pdbs", folder_with_pdbs, 
                "--out_folder", mpnn_dir, 
                "--chain_list", chains_to_design, 
                "--position_list", position_fixed, 
                "--num_seq_per_target", str(num_seq_per_target), 
                "--sampling_temp", str(sampling_temp), 
                "--seed", str(seed), 
                "--batch_size", str(num_seq_per_target), 
                "--omit_AAs", "C"])

In [0]:
df_fixed_positions = pd.read_csv(f"{volume_save_path}/filtered_links/fixed_positions.csv")
# Fixed Positions Correspond to FMWK positions as indicated in output
# CDR Positions missing from output
# Needs to be done after backbone generation because even tho, only using IMGT annotation. CDR3 length is randomly sampled for each backbone and not fixed
print(df_fixed_positions.iloc[0]['motif_index'])
print("-"*20)
print(df_fixed_positions.iloc[1]['motif_index'])

In [0]:
seq_file = f"{mpnn_dir}/seqs/{name}_0.fa"
!cat {seq_file} 

## Step 3: Use Flowpacker to add side chains to backbones 
Side chains determined from seqs derived/generated via AbMPNN from Step 2
Addresses issue that backbones must have sidechains prior to being evaluated via AF3Score


FlowPacker is first used to add side chains onto the backbones designed by PPIFlow, and the resulting full-atom structures are then evaluated using AF3Score for structural and interface quality assessment. 

Flowpacker is essentially used to create PDBs of the MPNN seqs by rather than predicting the new seq via AF2/Boltz2, use the MPNN derived seqs to identify side chains for each AA position, add them to the PPIFlow output (backbone only structure), and so now have structures for the MPNN-derived seqs that can be scored in the following step

In [0]:
#@title 3. Flowpacker Sidechain packing

import os
import csv
from Bio import SeqIO

def direct_fasta_to_csv(input_dirs: list, output_csv: str, suffix: str = ".pdb"):

    seen_seqs = set()

    os.makedirs(os.path.dirname(os.path.abspath(output_csv)), exist_ok=True)

    with open(output_csv, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(["link_name", "seq", "seq_idx"])
        for folder in input_dirs:
            if not os.path.exists(folder): continue
            files = [os.path.join(folder, f) for f in os.listdir(folder) if f.endswith(('.fasta', '.fa'))]

            for file_path in files:
                base_name = os.path.splitext(os.path.basename(file_path))[0]

                for i, record in enumerate(SeqIO.parse(file_path, "fasta")):
                    if i == 0:
                        continue

                    seq_str = str(record.seq)
                    if seq_str in seen_seqs:
                        continue
                    seen_seqs.add(seq_str)
                    link_name = f"{base_name}{suffix}"
                    seq_idx = str(i)

                    writer.writerow([link_name, seq_str, seq_idx])

    print(f"✅ Processing complete! {len(seen_seqs)} unique sequences have been written to: {output_csv}")



fa_folders = [
    f'{mpnn_dir}/seqs',
]

direct_fasta_to_csv(
    input_dirs=fa_folders,
    output_csv=f'{mpnn_dir}/final_result.csv',
    suffix=".pdb"
)

In [0]:
# Above Function extracted all seqs from fasta files and wrote to csv
df_mpnn_seqs = pd.read_csv(f"{mpnn_dir}/final_result.csv")
df_mpnn_seqs.head()

In [0]:
import yaml
flowpacker_yaml_path = "../configs/base_flowpacker.yaml"

# Open YAML file as a dictionary
with open(flowpacker_yaml_path, 'r') as file:
    yaml_dict = yaml.safe_load(file)
print(yaml_dict)
# Modify test_path argument in yaml file
yaml_dict['data']['test_path'] = f"{volume_save_path}/filtered_links"
yaml_dict['ckpt'] = "/Volumes/sandbox/model_weights/protein_hunter/bc40.pth"

# Save dictionary back to YAML file
with open(flowpacker_yaml_path, 'w') as file:
    yaml.safe_dump(yaml_dict, file, default_flow_style= False)

In [0]:
import sys
import subprocess

# Create Folder with the flowpacker outputs within volume save path
folder_name_flowpacker_output = f"{volume_save_path}/flowpacker_out"
os.makedirs(folder_name_flowpacker_output, exist_ok=True)

# Define Flowpacker Arguments and run flowpacker script (sampler_pdb_colab.py)
use_gt_masks = True
pdb_dir = filtered_dir
save_dir = folder_name_flowpacker_output
csv_file = f"{mpnn_dir}/final_result.csv"
# Run actual protein_mpnn_run script
subprocess.run(["python", "../demo_scripts/flowpacker_af3score/sampler_pdb_colab.py", flowpacker_yaml_path, 
                "--use_gt_masks", str(use_gt_masks),  
                "--pdb_dir", pdb_dir, 
                "--save_dir", folder_name_flowpacker_output,
                "--csv_file", csv_file, 
                ])

## Step 4: Score Designs which now have side chain atoms

2 Approaches:
1. Score the complex with its added side chains on its own via AF3Score
    * Output will preserve the original chain IDs of the input
2. Extract seqs for the complex and then run the seqs through Boltz2 (Using Approach #2 due to AF3 Commercial Restrictions)
    * Output will always have chain ID (A for Binder) &  (B for Target) -> which may differ from original input

In [0]:
import os
import subprocess

# The path where the package's setup.py or pyproject.toml is located
boltz_dir = "/Workspace/Users/karthik.raj@bio-techne.com/boltz" 

command = [
    "pip", "install", 
    "--no-dependencies", 
    str(boltz_dir)
]

subprocess.run(command, check=True)

In [0]:
import sys
import os
import shutil
import tarfile
from pathlib import Path

# --- CONFIGURATION ---
# Permanent Storage (Unity Catalog Volume)
VOLUME_DIR = Path("/Volumes/sandbox/model_weights/protein_hunter")
# Fast Local Storage (Ephemeral SSD)
LOCAL_DIR = Path.home() / ".boltz"

# Create directories
os.makedirs(VOLUME_DIR, exist_ok=True)
if os.path.exists(LOCAL_DIR):
    # Clean slate for local dir to avoid half-extracted messes
    if os.path.islink(LOCAL_DIR):
        os.remove(LOCAL_DIR)
    else:
        shutil.rmtree(LOCAL_DIR)
os.makedirs(LOCAL_DIR, exist_ok=True)

print(f"📍 Hybrid Setup Initiated")
print(f"   Storage: {VOLUME_DIR}")
print(f"   Execution: {LOCAL_DIR}")

# --- HELPER FUNCTIONS ---
def install_from_volume():
    print("🔄 Found files in Volume! Linking to local SSD...")
    
    # 1. Link the Weights (Instant)
    # We don't copy the 3GB file, we just symlink it.
    vol_ckpt = VOLUME_DIR / "boltz2_conf.ckpt"
    local_ckpt = LOCAL_DIR / "boltz2_conf.ckpt"
    if vol_ckpt.exists():
        if os.path.exists(local_ckpt): os.remove(local_ckpt)
        os.symlink(vol_ckpt, local_ckpt)
        print("   ✅ Weights linked.")
    
    # 2. Extract the Molecules (Fast on SSD)
    # We copy the tarball and extract it locally
    vol_tar = VOLUME_DIR / "mols.tar"
    local_tar = LOCAL_DIR / "mols.tar"
    
    if vol_tar.exists():
        print("   ⏳ Copying CCD tarball to local SSD...")
        shutil.copy(vol_tar, local_tar)
        
        print("   📦 Extracting molecules locally (this will be fast)...")
        with tarfile.open(local_tar) as tar:
            tar.extractall(path=LOCAL_DIR)
        print("   ✅ CCD Data ready.")

def download_and_backup():
    print("⬇️ Files missing in Volume. Downloading fresh...")
    
    # Setup path to import the downloader
    repo_path = os.getcwd()
    sys.path.insert(0, os.path.join(repo_path, "boltz_ph"))
    try:
        from boltz.main import download_boltz2
    except ImportError:
        sys.path.insert(0, os.path.join(repo_path, "boltz_ph", "src"))
        from boltz.main import download_boltz2

    # Download to LOCAL SSD first (Fastest)
    download_boltz2(LOCAL_DIR)
    
    print("💾 Backing up to Volume for future runs...")
    # Copy the big files to the volume
    local_ckpt = LOCAL_DIR / "boltz2_conf.ckpt"
    local_tar = LOCAL_DIR / "mols.tar" # The downloader usually keeps the tar?
    
    # Note: boltz downloader extracts immediately. We might need to re-tar or just check what's there.
    # Usually it leaves boltz2_conf.ckpt and mols/ folder.
    
    # Backup Weights
    if local_ckpt.exists():
        shutil.copy(local_ckpt, VOLUME_DIR / "boltz2_conf.ckpt")
        print("   ✅ Weights backed up.")
        
    # Backup Molecules (Re-tar them for efficient storage)
    if (LOCAL_DIR / "mols").exists():
        print("   zzZ Compressing molecules for backup...")
        with tarfile.open(VOLUME_DIR / "mols.tar", "w") as tar:
            tar.add(LOCAL_DIR / "mols", arcname="mols")
        print("   ✅ CCD Data backed up.")

# --- EXECUTION LOGIC ---
# Check if the Volume already has the goods
if (VOLUME_DIR / "boltz2_conf.ckpt").exists() and (VOLUME_DIR / "mols.tar").exists():
    install_from_volume()
else:
    download_and_backup()

print("\n🚀 Ready to run! Environment is optimized for Databricks.")

In [0]:
from StrucTools import *
from RunBoltz import *
import biotite.structure as struc
import biotite.sequence as seq
from tqdm import tqdm

path_flowpacker_outputs = f"{volume_save_path}/after_pdbs"
path_boltz_predictions = f"{volume_save_path}/after_pdbs/boltz_predictions"

all_boltz_metrics = []
for struc_filename in tqdm(os.listdir(path_flowpacker_outputs), desc="Boltz2 predictions"):
    if (struc_filename.endswith(".pdb") or struc_filename.endswith(".cif")):
        full_pdb_path = os.path.join(path_flowpacker_outputs, struc_filename)
        
        # 1. Extract atom array to ultimately extract seqs
        atom_array = extract_atom_array(struc_file_path= full_pdb_path)
        seqs, _ = struc.to_sequence(atom_array)
        seq_binder, seq_target = str(seqs[0]), str(seqs[1])
        print(seq_binder, seq_target)
        print("---" * 20)

        # 2. Create inputs to run boltz (Future should have this be an initial & update function of RunBoltz.py)
        seq_list = [seq_binder, seq_target]
        msa_options = ['empty', '']
        template_paths = [''] * len(seq_list)
        entity_type = ['protein'] * len(seq_list)
        desired_epitope_residues = []
        num_models = 5
        kernels = False
        design_name = struc_filename.split('.')[0]
        print("Design Name: ", design_name)

        # 3. Run Boltz Prediction
        df_design_metrics = boltz_predict_analyze(design_name=design_name, volume_save_path= path_boltz_predictions, seq_list=seq_list,
                                          msa_options=msa_options,template_paths=template_paths, entity_type=entity_type, desired_epitope_residues=desired_epitope_residues, num_models = num_models, kernels = kernels)
        all_boltz_metrics.append(df_design_metrics)

# 4. Concatenate all boltz predictions into one df
df_boltz_metrics = pd.concat(all_boltz_metrics)
df_boltz_metrics.head()

In [0]:
import biotite.structure.io.pdb as pdb
threshold_iptm = 0.8
df_boltz_metrics_most_confident_predictions = df_boltz_metrics[df_boltz_metrics['model_id'] == 0]
df_boltz_top_designs = df_boltz_metrics_most_confident_predictions[df_boltz_metrics_most_confident_predictions['iptm'] > threshold_iptm]
print(f"Number of top designs: {len(df_boltz_top_designs)} out of {len(df_boltz_metrics_most_confident_predictions)}")

# Create new directory to save top designs
path_boltz_top_predictions = f"{volume_save_path}/after_pdbs/boltz_predictions/filtered_links"
os.makedirs(path_boltz_top_predictions, exist_ok=True)

# Save top designs as PDB Files to new directory
for index, row in df_boltz_top_designs.iterrows():
    path_top_design = row['path_structure']
    path_new_save_destination = os.path.join(path_boltz_top_predictions, f"{row['design_name']}.pdb")
    # Extract Top Design CIF Path and convert to PDB File at new save destination
    atom_array = extract_atom_array(struc_file_path= path_top_design)
    pdb_file = pdb.PDBFile()
    pdb_file.set_structure(atom_array)
    pdb_file.write(path_new_save_destination)
    atom_array = extract_atom_array(struc_file_path= path_new_save_destination)
    seqs, _ = struc.to_sequence(atom_array)
    seq_binder, seq_target = str(seqs[0]), str(seqs[1])
    print(seq_binder, seq_target)


## Step 5: Rosetta Interface Analysis

Identify fixed residues based on having Rosetta Energy Units < -5

In [0]:
%pip install pyrosettacolabsetup pyrosetta-installer altair

In [0]:
import pyrosetta_installer
pyrosetta_installer.install_pyrosetta()

In [0]:
import biotite
import biotite.structure as struc
import biotite.sequence as seq
import os
import re
import subprocess
from StrucTools import *

interface_dist = 10
energy_filter = -5

pdb_folder = path_boltz_top_predictions
output_dir = f"{volume_save_path}/iptm_filter_rst" 
os.makedirs(output_dir, exist_ok=True)

# find pdbs in folder
pdb_files = []
for file in os.listdir(pdb_folder):
    if file.endswith(".pdb"):
        pdb_files.append(file)
        print(file)

# update xml file
resnums = "" # should include all positions "1A-119A,1M-61M"
xml_path = f"../calculate_rosetta_energies/update.xml"

# open pdb to extract residues
for pdb in pdb_files:
    # 1. Create appropriate resnum: PDB chain_res_start{chain_id} - PDB chain_res_end{chain_id}". Differs based on length of CDR3
    pdb_path = f"{pdb_folder}/{pdb}"
    atom_array = extract_atom_array(struc_file_path= pdb_path)
    seqs, _ = struc.to_sequence(atom_array)
    len_binder, len_target = len(str(seqs[0])), len(str(seqs[1]))
    chains = struc.get_chains(atom_array)
    resnums = f"1{chains[0]}-{len_binder}{chains[0]},1{chains[1]}-{len_target}{chains[1]}"

    # 2. Update xml file: specifically the resnums it is looking at
    # 2.1 Read the entire XML file as a single string
    with open(xml_path, "r") as f:
        content = f.read()

    # 2.2 Use Regex to find and replace the resnums attribute
    # This looks for exactly 'resnums="' followed by anything that isn't a quote, ending with '"'
    resnums_pattern = r'resnums="[^"]+"'
    new_content = re.sub(resnums_pattern, f'resnums="{resnums}"', content)

    # 2.3 Safely write the updated string back to the file
    with open(xml_path, "w") as f:
        f.write(new_content)

    print(f"Successfully updated resnums to: {resnums}")
    
    # run energy calculation
    comand = ["python", f"../calculate_rosetta_energies/per_residue_energy_pyrosetta.py",
              "--pdb", f"{pdb_folder}/{pdb}",
              "--binder_id", "A",
              "--target_id", "B",
              "--interface_dist", str(interface_dist),
              "--output_dir", output_dir,
              "--xml_protocol", xml_path]
    subprocess.run(comand, check=True)

In [0]:
volume_save_path

In [0]:
import os
import ast
all_residue_energies = []
for file in os.listdir(output_dir):
    if "residue_energy_pyrosetta.csv" in file:
        residue_energy_path = f"{output_dir}/{file}"
        df_residue_energy = pd.read_csv(residue_energy_path)
        all_residue_energies.append(df_residue_energy)
df_residue_energy = pd.concat(all_residue_energies).reset_index(drop = True)
df_residue_energy.insert(loc = 1, column = 'target_id', value = 'B')
df_residue_energy.insert(loc = 1, column = 'pdbname', value = df_residue_energy['pdbpath'].apply(lambda filepath: (filepath.split('/')[-1]).split('.')[0]))
df_residue_energy

# Identify fixed residues per design and the relative frequency
df_residue_energy["fixed_residue"] = df_residue_energy["binder_energy"].apply(lambda x: "_".join([str(k) for k, v in ast.literal_eval(x).items() if v < -5]))
df_residue_energy["fixed_residue"] = df_residue_energy["fixed_residue"].astype(str)
df_residue_energy["num_fixed_residue"] = df_residue_energy["fixed_residue"].apply(
    lambda x: 0 if x == "" else len(x.split("_"))
)
print(df_residue_energy.shape)
path_complete_residue_energy = f"{volume_save_path}/rst_intface_residue_energy_with_key_contacts.csv"
df_residue_energy.to_csv(path_complete_residue_energy,index=False)
df_residue_energy

## Step 6: Affinity Maturation 

#### 6.1: Extract ideal residues from respective MPNN designs and merge into single PDB for corresponding input backbone

In [0]:
volume_save_path

In [0]:
import subprocess
path_folder_passed_iptm = f"{volume_save_path}/after_pdbs/boltz_predictions/filtered_links/"
path_passed_iptm_seqs = f"{path_folder_passed_iptm}/filtered_links.csv"
command = ["python", "../demo_scripts/pdb2fa.py", path_folder_passed_iptm, "csv", path_passed_iptm_seqs]
subprocess.run(command, check=True)

In [0]:
import ast
residue_energy = pd.read_csv(path_complete_residue_energy)
seq = path_passed_iptm_seqs
path_fixed_residue=f"{volume_save_path}/fixed_residue.csv"

seq_df_multi = pd.read_csv(seq)
seq_df_multi['pdb_name']=seq_df_multi['pdb_name'].str.replace('.pdb','')
residue_energy = residue_energy.rename(columns={'pdbname':'pdb_name'})

residue_energy = pd.merge(residue_energy, seq_df_multi, on="pdb_name")
residue_energy["structure"] = residue_energy["pdb_name"].apply(lambda x: '_'.join(x.split("_")[:-1]))
residue_energy["num_fixed_residue"] = residue_energy["fixed_residue"].apply(lambda x: len(x.split("_")) if not pd.isna(x) else 0)

all_cand = []
for i, group in residue_energy.groupby("structure"):
    key_res = {}
    for j, row in group.iterrows():

        inter_res = {str(k):[v, row["sequence"][int(k)-1]] for k, v in ast.literal_eval(row["binder_energy"]).items() if v < -5}

        for k in inter_res:
            if k not in key_res:
                key_res[k] = inter_res[k]
            elif inter_res[k][0] <= key_res[k][0]:
                key_res[k] = inter_res[k]


    all_cand.append([row["pdbpath"], "_".join(row["pdb_name"].split('_')[:-1]), row["target_id"], row["binder_id"], key_res, len(key_res)])

        # key_res = {k: inter_res[k] if inter_res[k][0] <= key_res[k][0] else key_res[k] for k in inter_res}
all_cand_df = pd.DataFrame(all_cand, columns=["pdb_path", "pdb_name", "target_id", "binder_id", "key_res", "num_fixed_residues"])
print(all_cand_df.shape)
all_cand_df.to_csv(f'{path_fixed_residue}',index=False)

In [0]:
all_cand_df

In [0]:
# merge key residues into one pdb file
import pandas as pd
import os
import ast
from tqdm import tqdm

def process_pdb_mutation_and_renumber(csv, pdb_output_dir, 
                                      renumber_chain='B',
                                      start_index=1):
    one_to_three = {
        'A': 'ALA', 'R': 'ARG', 'N': 'ASN', 'D': 'ASP', 'C': 'CYS',
        'Q': 'GLN', 'E': 'GLU', 'G': 'GLY', 'H': 'HIS', 'I': 'ILE',
        'L': 'LEU', 'K': 'LYS', 'M': 'MET', 'F': 'PHE', 'P': 'PRO',
        'S': 'SER', 'T': 'THR', 'W': 'TRP', 'Y': 'TYR', 'V': 'VAL'
    }
    
    df = pd.read_csv(csv)
    os.makedirs(pdb_output_dir, exist_ok=True)

    print(f"Start processing: Renaming Chain A & Renumbering Chain {renumber_chain} (start from {start_index})...")

    for idx, row in tqdm(df.iterrows(), total=df.shape[0]):
        
        if not os.path.exists(row["pdb_path"]):
            print(f"Warning: {row['pdb_path']} not found, skipping...")
            continue
        
        # No point creating a merged PDB if it has no fixed residues
        if row["num_fixed_residues"] == 0:
            print(f"Warning: No fixed residues found for {row['pdb_name']}, skipping...")
            continue

        resi_to_resname = {}
        if "key_res" in row and pd.notna(row["key_res"]):
            try:
                for resi, aa in ast.literal_eval(row["key_res"]).items():
                    resi_to_resname[int(resi)] = one_to_three.get(aa[1], "UNK")
            except Exception as e:
                print(f"Error parsing key_res for {row['pdb_name']}: {e}")

        with open(row["pdb_path"], "r") as f:
            pdb_lines = f.readlines()

        new_lines = []
        
        current_renumber_idx = start_index - 1
        last_seen_resi_id = None

        for line in pdb_lines:
            if not line.startswith("ATOM"):
                new_lines.append(line)
                continue

            chain_id = line[21]
            

            if chain_id == "A":
                resi = int(line[22:26])
                if resi in resi_to_resname:
                    new_resname = resi_to_resname[resi]

                    line = line[:17] + new_resname.ljust(3) + line[20:]

            elif chain_id == renumber_chain:
                current_resi_id_str = line[22:27]                
                if current_resi_id_str != last_seen_resi_id:
                    current_renumber_idx += 1
                    last_seen_resi_id = current_resi_id_str

                new_resi_str = f"{current_renumber_idx:>4}"
                line = line[:22] + new_resi_str + " " + line[27:]
            new_lines.append(line)
        output_pdb = os.path.join(pdb_output_dir, os.path.basename(row["pdb_name"])+".pdb")
        with open(output_pdb, "w") as f:
            f.writelines(new_lines)
    print("All done!")

path_save_merged_pdb = f"{volume_save_path}/merge_motif_pdb/"
process_pdb_mutation_and_renumber(path_fixed_residue, path_save_merged_pdb, 
    renumber_chain='B', start_index=9) # The residue numbering of the target chain is consistent with the input target file

In [0]:
import sys
sys.path.append('../demo_scripts')
from AntpackCDR import *

path_filter_cdr = f"{volume_save_path}/filter_cdr_idx.csv"

df_cdrs = extract_fmwk_cdr_indices_from_folder(path_save_merged_pdb, path_filter_cdr)
df_cdrs

In [0]:
import os
import ast
import pandas as pd

def safe_extract_keys(val):

    if pd.isna(val) or str(val).strip() == "":
        return ""
    try:
        d = ast.literal_eval(val)
        if isinstance(d, dict):
            return " ".join([str(k).strip() for k in d.keys()])
    except (ValueError, SyntaxError):
        return ""
    return ""

def process_indices(val, output_format="space"):

    if pd.isna(val) or str(val).strip() == "":
        return ""
    
    try:
        unique_sorted_ints = sorted(list(set(int(i) for i in str(val).split() if i.strip())))
        
        if output_format == "pdb":
            return ",".join([f"A{i}" for i in unique_sorted_ints])
        else:
            return " ".join([str(i) for i in unique_sorted_ints])
            
    except ValueError:
        return ""
# ==========================================
# 1. Load & Preprocess fw_csv
fw_csv = pd.read_csv(path_filter_cdr)
fw_csv = fw_csv[['pdb_name', 'fw_index', 'r2_cdr_pos']].copy()
fw_csv.rename(columns={'pdb_name': 'pdb_struc_name'}, inplace=True)

# 2. Load & Preprocess key_res_csv
key_res_csv = pd.read_csv(f"{volume_save_path}/fixed_residue.csv")
output_path = f"{volume_save_path}/fixed_residue_setting_for_partial_flow.csv"

key_res_csv["pdb_struc_name"] = key_res_csv["pdb_name"]#.apply(lambda x: '_'.join(str(x).split("_")[:-1]))

# 3. Merge DataFrames

fixed_positions_df = fw_csv.merge(key_res_csv, on="pdb_struc_name", how='inner')

# 4. Extract Key Residues
fixed_positions_df["key_res_index"] = fixed_positions_df["key_res"].apply(safe_extract_keys)

# 5. Combine Indices (Raw String Concatenation)

raw_combined_indices = (
    fixed_positions_df['key_res_index'].fillna("").astype(str) + 
    " " + 
    fixed_positions_df['fw_index'].fillna("").astype(str)
)

# 6. Generate Final Columns

fixed_positions_df['r2_fixed_sequence_pm'] = raw_combined_indices.apply(lambda x: process_indices(x, output_format="space"))

fixed_positions_df['r2_fixed_sequence'] = raw_combined_indices.apply(lambda x: process_indices(x, output_format="pdb"))

fixed_positions_df['r2_fixed_structure'] = fixed_positions_df['key_res_index'].apply(lambda x: process_indices(x, output_format="pdb"))

os.makedirs(os.path.dirname(output_path), exist_ok=True)
fixed_positions_df.to_csv(output_path, index=False)

print(f"Data processed and saved to {output_path}")
print(fixed_positions_df[['r2_fixed_sequence_pm', 'r2_fixed_sequence']].head())

#### 6.2: Partial Flow on MPNN Backbones with Fixed Residues Across MPNN Derived Seqs for Given Backbone

In [0]:
import os
import shutil
import subprocess 

BASE = volume_save_path
input_dir = f"{BASE}/merge_motif_pdb"
output_dir = f"{BASE}/pf"  # Final destination on Volume
local_output_dir = "/tmp/pf"  # Local temp dir for pipeline writes
#hotspot= "B28,B39,B55,B61"
hotspot = specified_hotspots
shelldir = f"{output_dir}/submit"
logdir = f'{output_dir}/log'
os.makedirs(output_dir, exist_ok=True)
os.makedirs(local_output_dir, exist_ok=True)
os.makedirs(shelldir, exist_ok=True)
os.makedirs(logdir, exist_ok=True)

binder_chain, target_chain = ["A","B"]
samples_per_target=8
t_start, t_end = [0.6,0.6]
interface_dist = 10
model_weights = "/Volumes/sandbox/model_weights/protein_hunter/nanobody.ckpt"

fixed_positions_df = pd.read_csv(f"{BASE}/fixed_residue_setting_for_partial_flow.csv")
for i, row in fixed_positions_df.iterrows():

    pdb = os.path.join(input_dir, row["pdb_name"]+".pdb")
    pdb_name = row["pdb_name"]
    fixed_structure = row["r2_fixed_structure"]
    cdr_pos = row["r2_cdr_pos"]

    # If already exists on Volume, remove and run again
    volume_dest = f"{output_dir}/{pdb_name}"
    print(volume_dest)
    if os.path.exists(volume_dest):
        shutil.rmtree(volume_dest)

    # Use local temp path for pipeline writes (avoids FUSE append-mode error)
    local_dest = f"{local_output_dir}/{pdb_name}"
    if os.path.exists(local_dest):
        shutil.rmtree(local_dest)
    
    command = ["python", "../sample_antibody_nanobody_partial.py",
                "--complex_pdb", pdb,
                "--start_t", str(t_start),
                "--fixed_positions", fixed_structure,
                "--antigen_chain", target_chain,
                "--heavy_chain", binder_chain,
                "--config", "../configs/inference_nanobody.yaml",
                "--samples_per_target", str(samples_per_target),
                "--cdr_position", cdr_pos,
                "--specified_hotspots", hotspot,
                "--retry_Limit", "10",
                "--model_weights", model_weights,
                "--output_dir", local_dest,
                "--name", pdb_name]
    result = subprocess.run(command)

    # Copy results from local temp dir back to Volume
    if result.returncode == 0 and os.path.exists(local_dest):
        shutil.copytree(local_dest, volume_dest)
        print(f"Copied {pdb_name} results to Volume.")
    elif result.returncode != 0:
        print(f"Pipeline failed for {pdb_name} with return code {result. returncode}")

# Clean up local temp dir
if os.path.exists(local_output_dir):
    shutil.rmtree(local_output_dir)
    print("Cleaned up local temp directory.")

In [0]:
import os
import shutil
from pathlib import Path

src_root = Path(f"{volume_save_path}/pf")
round2_backbone_dir = src_root / "link_samples"
round2_backbone_dir.mkdir(exist_ok=True)

for subdir in src_root.iterdir():
    if not subdir.is_dir():
        continue
    if subdir.name == "link_samples":
        continue

    for pdb in subdir.glob("sample*.pdb"):

        idx = pdb.stem.replace("sample", "")

        new_name = f"{subdir.name}_{idx}.pdb"
        dst_path = round2_backbone_dir / new_name
        if not dst_path.exists():
            shutil.copy2(pdb, dst_path)

### 6.3: MPNN Non-Rosetta Fixed Residues Redesign of Partial Flow Backbones

In [0]:
fixed_positions_df

In [0]:
import os
import subprocess
import time
import pandas as pd

# Initial Setup
input_dir = f"{volume_save_path}/pf"
output_dir = f"{volume_save_path}/round2_mpnn"
resi = pd.read_csv(f'{volume_save_path}/fixed_residue_setting_for_partial_flow.csv')

# Log and Submit directories aren't strictly needed if we aren't using SLURM, 
# but keeping them if you still want the folder structure.
shelldir = f"{output_dir}/submit"
logdir = f'{output_dir}/logs'
os.makedirs(output_dir, exist_ok=True)
os.makedirs(shelldir, exist_ok=True)
os.makedirs(logdir, exist_ok=True)

num_seq_per_target = 4
sampling_temp = 0.1
batch_size = num_seq_per_target
omit_AAs = "C"
chains_to_design = "A"

for i, folder in enumerate(os.listdir(input_dir)):
    # Handle case of link_samples folder creation. Not exactly sure on its utility
    if PREFIX not in folder:
        continue

    print(f"\nProcessing folder: {folder}")
    pdb_name = folder
    folder_with_pdbs = os.path.join(input_dir, folder)
    # Create separate folder for mpnn_seq outputs
    output_dir_temp = os.path.join(output_dir, folder)
    os.makedirs(output_dir_temp, exist_ok=True)
    
    # Ensure this is cast to a string just in case pandas returns a numeric type
    fixed_positions = resi.loc[resi['pdb_name'] == pdb_name, 'r2_fixed_sequence_pm'].values[0]
    # Save fixed_positions into CSV for compatability with custom_script
    formatted_df = pd.DataFrame({'PDB_ID': resi['pdb_name'],'fixed_positions': resi['r2_fixed_sequence_pm']})
    positions_list_path = os.path.join(output_dir_temp,'fixed_positions_mpnn.csv')
    formatted_df.to_csv(positions_list_path, index = False)

    # Define output paths
    path_for_parsed_chains = os.path.join(output_dir_temp, 'parsed_pdbs.jsonl')
    path_for_assigned_chains = os.path.join(output_dir_temp, 'assigned_pdbs.jsonl')
    path_for_fixed_positions = os.path.join(output_dir_temp, 'fixed_pdbs.jsonl')

    # 1. Parse multiple chains
    cmd1 = [
        "python", "../ProteinMPNN/helper_scripts/parse_multiple_chains.py",
        "--input_path", folder_with_pdbs,
        "--output_path", path_for_parsed_chains
    ]

    # 2. Assign fixed chains
    cmd2 = [
        "python", "../ProteinMPNN/helper_scripts/assign_fixed_chains.py",
        "--input_path", path_for_parsed_chains,
        "--output_path", path_for_assigned_chains,
        "--chain_list", chains_to_design
    ]

    # 3. Make fixed positions dict
    cmd3 = [
        "python", "../ProteinMPNN/helper_scripts/make_fixed_positions_dict_og.py",
        "--input_path", path_for_parsed_chains,
        "--output_path", path_for_fixed_positions,
        "--chain_list", chains_to_design,
        "--position_list", fixed_positions
    ]

    # 4. Run ProteinMPNN
    cmd4 = [
        "python", "../ProteinMPNN/protein_mpnn_run.py",
        "--path_to_model_weights", "/Volumes/sandbox/model_weights/protein_hunter",
        "--folder_with_pdbs_path", folder_with_pdbs,
        "--model_name", "abmpnn",
        "--jsonl_path", path_for_parsed_chains,
        "--chain_id_jsonl", path_for_assigned_chains,
        "--fixed_positions_jsonl", path_for_fixed_positions,
        "--out_folder", output_dir_temp,
        "--num_seq_per_target", str(num_seq_per_target),
        "--sampling_temp", str(sampling_temp),
        "--batch_size", str(batch_size),
        "--use_soluble_model",
        "--omit_AAs", omit_AAs
    ]

    commands = [cmd1, cmd2, cmd3, cmd4]
    
    start_time = time.time()
    print(f"Job started at: {time.ctime()}")

    try:
        # Run each command sequentially
        for step, cmd in enumerate(commands, 1):
            print(f"  -> Running step {step}/4...")
            # check=True raises a CalledProcessError if the command fails
            subprocess.run(cmd, check=True)
            
        elapsed = time.time() - start_time
        print(f"Job ended at: {time.ctime()}")
        print(f"Elapsed time: {int(elapsed)} seconds")

    except subprocess.CalledProcessError as e:
        print(f"Error: Step {step} failed for {pdb_name} with return code {e.returncode}.")
        # Optionally break or continue based on whether you want a failure to stop the whole script
        continue

## Step 7: Score all Partial Flow -> MPNN Designs via Boltz, Rosetta, DockQ

In [0]:
import os
from pathlib import Path
from multiprocessing import Pool, cpu_count
from tqdm.auto import tqdm
import pandas as pd

def _parse_single_fasta(fasta_path: Path) -> list:

    records = []
    path_str = str(fasta_path)
    try:
        with fasta_path.open("r", encoding="utf-8") as fh:
            seq_idx = -1
            seq_buf = []

            for line in fh:
                if line.startswith(">"):
                    if seq_buf:
                        records.append([path_str, "".join(seq_buf), seq_idx])
                        seq_buf = []
                    seq_idx += 1
                else:

                    stripped_line = line.strip()
                    if stripped_line:
                        seq_buf.append(stripped_line)

            if seq_buf:
                records.append([path_str, "".join(seq_buf), seq_idx])
    
    except Exception as e:
        print(f"[WARN] Skipping file {fasta_path} due to error: {e}")
        return []

    return records

def read_all_sequences_parallel(input_dir: str, output_csv: str) -> pd.DataFrame:
    fasta_paths = list(Path(input_dir).rglob("*.fa"))
    if not fasta_paths:
        print("Warning: No .fa files found in the specified directory.")
        df = pd.DataFrame(columns=["fasta", "seq", "seq_idx"])
        df.to_csv(output_csv, index=False)
        return df
    
    print(f"Found {len(fasta_paths)} files. Starting parallel parsing...")

    num_processes = cpu_count()
    all_records = []

    with Pool(processes=num_processes) as pool:
        with tqdm(total=len(fasta_paths), desc="Parsing FASTA files in parallel") as pbar:
            for records_from_one_file in pool.imap_unordered(_parse_single_fasta, fasta_paths):
                if records_from_one_file:
                    all_records.extend(records_from_one_file)
                pbar.update(1)

    if not all_records:
        print("Warning: All files are empty or could not be parsed; the resulting CSV will be empty.")

    print("\nAll files have been parsed. Creating DataFrame and saving to CSV...")
    df = pd.DataFrame(all_records, columns=["fasta", "seq", "seq_idx"])
    df.to_csv(output_csv, index=False)
    print(f"Successfully saved {len(df)} sequences to {output_csv}")
    
    return df


input_directory = f"{volume_save_path}/round2_mpnn"
output_csv = f"{volume_save_path}/round2_mpnn_all.csv"
sequences_df = read_all_sequences_parallel(input_directory, output_csv)
sequences_df['seq_idx']= '' + sequences_df['seq_idx'].astype(str)
sequences_df.to_csv(output_csv,index=False)

In [0]:
sequences_df.head()

In [0]:
import os
import csv
from Bio import SeqIO

def direct_fasta_to_csv(input_dirs: list, output_csv: str, suffix: str = ".pdb"):

    seen_seqs = set()

    os.makedirs(os.path.dirname(os.path.abspath(output_csv)), exist_ok=True)

    with open(output_csv, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(["backbone_name", "link_name", "seq", "seq_idx"])
        for folder in input_dirs:
            if not os.path.exists(folder): continue
            files = [os.path.join(folder, f) for f in os.listdir(folder) if f.endswith(('.fasta', '.fa'))]

            # Add backbone name
            backbone_name = folder.split('/seqs/')[0].split('/')[-1]
            print(backbone_name)

            for file_path in files:
                base_name = os.path.splitext(os.path.basename(file_path))[0]
                print(file_path)

                

                for i, record in enumerate(SeqIO.parse(file_path, "fasta")):
                    if i == 0:
                        continue

                    seq_str = str(record.seq)
                    if seq_str in seen_seqs:
                        continue
                    seen_seqs.add(seq_str)
                    link_name = f"{base_name}{suffix}"
                    seq_idx = str(i)

                    writer.writerow([f"{backbone_name}_{link_name.split('.')[0]}", link_name, seq_str, seq_idx])

    print(f"✅ Processing complete! {len(seen_seqs)} unique sequences have been written to: {output_csv}")


dir_mpnn = f'{volume_save_path}/round2_mpnn'
#fa_folders = [
#    "/Volumes/sandbox/denovotrial/ppiflow/Mar19_4dn4_ccl2/round2_mpnn/Mar19_4dn4_ccl2_1/seqs/",
#    "/Volumes/sandbox/denovotrial/ppiflow/Mar19_4dn4_ccl2/round2_mpnn/Mar19_4dn4_ccl2_2/seqs/"
#]
fa_folders = [f"{dir_mpnn}/{folder_name}/seqs/" for folder_name in os.listdir(dir_mpnn) if PREFIX in folder_name]

direct_fasta_to_csv(
    input_dirs=fa_folders,
    output_csv=f'{dir_mpnn}/final_result.csv',
    suffix=".pdb"
)

In [0]:
df_round2_mpnn_seqs = pd.read_csv(f"{dir_mpnn}/final_result.csv")
df_round2_mpnn_seqs.head(10)

In [0]:
from StrucTools import *
import biotite.structure as struc
pdb_input = f"{volume_save_path}/{name}_0.pdb"
atom_array = extract_atom_array(pdb_input)
seqs, chain_starts = struc.to_sequence(atom_array)
binder_seq, target_seq = str(seqs[0]), str(seqs[1])
binder_seq, target_seq

In [0]:
from StrucTools import *
from RunBoltz import *
import biotite.structure as struc
import biotite.sequence as seq

path_boltz_predictions = f"{dir_mpnn}/boltz_predictions"

all_boltz_metrics = []
for index, row in df_round2_mpnn_seqs.iterrows():

    # 1. Create inputs to run boltz (Future should have this be an initial & update function of RunBoltz.py)
    seq_binder = row['seq']
    seq_list = [seq_binder, target_seq]
    msa_options = ['empty', '']
    template_paths = [''] * len(seq_list)
    entity_type = ['protein'] * len(seq_list)
    desired_epitope_residues = []
    num_models = 5
    kernels = False
    # Update struc filename.
    design_name = f"{row['backbone_name']}_{row['seq_idx']}"
    print("Design Name: ", design_name)

    # 3. Run Boltz Prediction
    df_design_metrics = boltz_predict_analyze(design_name=design_name, volume_save_path= path_boltz_predictions, seq_list=seq_list, msa_options=msa_options,template_paths=template_paths, entity_type=entity_type, desired_epitope_residues=desired_epitope_residues, num_models = num_models, kernels = kernels)
    
    all_boltz_metrics.append(df_design_metrics)

# 4. Concatenate all boltz predictions into one df
df_boltz_metrics = pd.concat(all_boltz_metrics)
df_boltz_metrics.to_csv(f"{path_boltz_predictions}/round2_mpnn_boltz.csv", index = False)
df_boltz_metrics.head()

In [0]:
df_boltz_metrics